<a href="https://colab.research.google.com/github/gav-ip/ML-zero/blob/main/pytorch_of_transformer_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U flax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 32.8 MB/s eta 0:00:00
  Attempting uninstall: flax
    Found existing installation: flax 0.11.2
    Uninstalling flax-0.11.2:
      Successfully uninstalled flax-0.11.2


In [ ]:
import jax
from jax import nn as jnn
from jax import random
import jax.numpy as jnp
import flax.nnx as nnx
import numpy as np
import random as py_random
import optax

In [ ]:
USE_GPU = True
device = jax.devices("cpu")[0] if not USE_GPU else jax.devices()[0]
block_size = 11
batch_size = 512 # Increased for potentially faster training, monitor memory
eval_iters = 200
max_iters = 1000 # Reduced for faster experimentation
eval_interval = 100 # Increased to reduce evaluation frequency
n_embed = 16 # Reduced for faster computation
n_heads = 2 # Reduced for faster computation
n_layer = 3 # Reduced for faster computation
dropout = 0.2
learning_rate = 1e-2 # Can be adjusted with batch_size, but requires tuning

print(device)

cuda:0


In [3]:
vocab = {0:'0', 1:'1', 2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8', 9:'9', 10:'+', 11:'=', 12:' '}
vocab_size = len(vocab)

In [4]:
stoi = {vocab[i]:i for i, ch in enumerate(vocab)}
itos = {i:vocab[i] for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[int(i)] for i in l])

print(encode("8+2 =10 "))
print(decode(encode("8+7 =15 ")))

[8, 10, 2, 12, 11, 1, 0, 12]
8+7 =15 


In [5]:
!wget https://raw.githubusercontent.com/gav-ip/ML-zero/main/transformer-adder/addition.txt

--2026-11-19 01:44:08--  https://raw.githubusercontent.com/gav-ip/ML-zero/main/transformer-adder/addition.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1300000 (1.2M) [text/plain]
Saving to: ‘addition.txt’

addition.txt        100%[===================>]   1.24M  --.-KB/s    in 0.03s   

2026-11-19 01:44:08 (39.7 MB/s) - ‘addition.txt’ saved [1300000/1300000]



In [8]:
with open("addition.txt", "r", encoding="utf-8") as f:
    text = [line.rstrip("\n") for line in f]

data = torch.tensor(encode(text), dtype=torch.long)
data[:3]

KeyError: '850+660=0151'

In [ ]:
print(data[:5])
print(decode(data[0]))

[[ 8  5  0 10  6  6  0 11  0  1  5  1]
 [ 2  2  7 10  8  5  7 11  4  8  0  1]
 [ 7  1  7 10  5  4  6 11  3  6  2  1]
 [ 7  4  0 10  9  1  4 11  4  5  6  1]
 [ 3  7  8 10  2  5  2 11  0  3  6 12]]
850+660=0151


In [ ]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
def get_batch(key, split):
    data = train_data if split == 'train' else val_data

    ix = random.randint(key, (batch_size,), 0, len(data))
    seq = data[ix]

    x = seq[:, :]
    y = seq[:, 1:]
    y = y.at[:,:7].set(-1)

    return x, y

xb, yb = get_batch(random.PRNGKey(0), 'train')
print('inputs:')
print(xb.shape)
print(decode(xb[0]))
print('targets:')
print(yb.shape)
print((yb[0]))

inputs:
(512, 12)
826+941=7671
targets:
(512, 11)
[-1 -1 -1 -1 -1 -1 -1  7  6  7  1]


In [ ]:
rngs = nnx.Rngs(0)

class MultiHeadAttention(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.key = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.query = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.value = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.proj = nnx.Linear(n_embed, n_embed, rngs=rngs)
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

    def __call__(self, x):
        B, T, C = x.shape
        head_dim = C // n_heads
        k = self.key(x).reshape(B, T, n_heads, head_dim)
        q = self.query(x).reshape(B, T, n_heads, head_dim)
        v = self.value(x).reshape(B, T, n_heads, head_dim)

        out = jnn.dot_product_attention(q, k, v, mask=None, is_causal=True)
        out = out.reshape(B, T, C)

        return self.dropout(self.proj(out))

class FeedForward(nnx.Module):
    def __init__(self, n_embed, *, rngs: nnx.Rngs):
        self.net = nnx.Sequential(
            nnx.Linear(n_embed, 4 * n_embed, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * n_embed, n_embed, rngs=rngs),
            nnx.Dropout(dropout, rngs=rngs),
        )

    def __call__(self, x):
        return self.net(x)

class Block(nnx.Module):
    def __init__(self, n_embd, n_head, *, rngs: nnx.Rngs):
        self.sa = MultiHeadAttention(rngs=rngs)
        self.ff = FeedForward(n_embd, rngs=rngs)
        self.ln1 = nnx.LayerNorm(n_embd, rngs=rngs)
        self.ln2 = nnx.LayerNorm(n_embd, rngs=rngs)

    def __call__(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class AdditionModel(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        super().__init__()
        self.token_embedding_table = nnx.Embed(vocab_size, n_embed, rngs=rngs)
        self.position_embedding_table = nnx.Embed(block_size, n_embed, rngs=rngs)
        self.blocks = nnx.Sequential(
            *[Block(n_embed, n_heads, rngs=rngs) for _ in range(n_layer)]
        )
        self.ln_f = nnx.LayerNorm(n_embed, rngs=rngs)
        self.lm_head = nnx.Linear(n_embed, vocab_size, rngs=rngs)

    def __call__(self, idx, targets=None):

        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(jnp.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)

            targets = targets.reshape(B*T)
            valid = targets != -1
            safe_targets = jnp.clip(targets, 0)

            token_loss = optax.softmax_cross_entropy_with_integer_labels(logits, safe_targets)
            loss = (token_loss * valid).sum() / jnp.maximum(valid.sum(), 1)

        return logits, loss

    def generate(self, idx, max_new_tokens, key):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            key, subkey = random.split(key)
            idx_next = jax.random.categorical(subkey, logits)

            # Reshape idx_next to (B, 1) to match dimensions for concatenation
            idx_next = jnp.expand_dims(idx_next, axis=1)

            idx = jnp.concatenate((idx, idx_next), axis=1)

        return idx

In [ ]:
model = AdditionModel(rngs=nnx.Rngs(0))

In [ ]:
key = random.PRNGKey(0)

# call jax.lax.stop_gradient for no_grad
def estimate_loss(key):
    out = {}
    model.eval()
    for split in ['train', 'val']:
         losses = jnp.zeros(eval_iters)
         for k in range(eval_iters):
             key, subkey = random.split(key)
             X, Y = get_batch(subkey, split)
             _, loss = model(X, Y)
             losses = losses.at[k].set(loss)
         out[split] = losses.mean()
    model.train()
    return out

In [ ]:
params = nnx.state(model, nnx.Param)
param_count = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total parameters: {param_count}")

Total parameters: 10333


In [ ]:
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=learning_rate), wrt=nnx.Param)

@jit
def compute_train_loss(model, x, y):
    _, loss_value = model(x, y)
    return loss_value

# training loop
key = random.PRNGKey(1)

for steps in range(max_iters):
    key, eval_key, batch_key = random.split(key, 3)

    if steps % eval_interval == 0:
        losses = estimate_loss(eval_key)
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # get sample batch from training set
    xb, yb = get_batch(batch_key,'train')

    # Evaluate loss and compute gradients
    # The first return value is the scalar loss, the second are the gradients
    _, grads = nnx.value_and_grad(compute_train_loss)(model, xb, yb)
    optimizer.update(model, grads)

step 0: train loss 3.1113, val loss 3.1099
step 100: train loss 1.0497, val loss 1.0495
step 200: train loss 0.6332, val loss 0.6306
step 300: train loss 0.2415, val loss 0.2328
step 400: train loss 0.1044, val loss 0.1041
step 500: train loss 0.1023, val loss 0.1049
step 600: train loss 0.0716, val loss 0.0706
step 700: train loss 0.0248, val loss 0.0254
step 800: train loss 0.0323, val loss 0.0338
step 900: train loss 0.0284, val loss 0.0300


In [ ]:
import pickle

checkpoint = {
    'model_state_dict': nnx.state(model, nnx.Param), # Corrected to use nnx.state
    'stoi': stoi,
    'itos': itos,
    'config': {
        'vocab_size': vocab_size,
        'block_size': block_size,
        'n_embd': n_embed,
        'n_head': n_heads,
        'n_layer': n_layer,
        'dropout': dropout,
    }
}
with open('addition_model.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)

In [ ]:
with open('addition_model.pkl', 'rb') as f:
        ckpt = pickle.load(f)

cfg = ckpt['config']
stoi, itos = ckpt['stoi'], ckpt['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda a: ''.join([itos[int(i)] for i in a])

m = AdditionModel(rngs=nnx.Rngs(0))
nnx.update(m, ckpt['model_state_dict'])
m.eval()

In [ ]:
prompt = "636+126="
idx = jnp.array([encode(prompt)], device=device)
out = m.generate(idx, max_new_tokens=4, key=random.PRNGKey(0))
result = decode(out[0])
res = result[-5:]
reversed_res = res[::-1]

print(result)
print(reversed_res)

right now we are overfitting, we need to implement some form of
early stopping